<a href="https://colab.research.google.com/github/norewyx0205/vlm-event-boundary/blob/main/notebooks/colab_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ladder Event Boundary Evaluation on Colab

This notebook keeps the baseline sanity-check evaluation and runs the 6-level ladder experiment with Qwen3-VL.


In [5]:
%cd /content
!ls

/content
sample_data  vlm-event-boundary


In [6]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")

## Clone or Update Repository

If the repository already exists in Colab, this cell pulls the latest code. If it does not exist, it clones the repo.


In [7]:
from getpass import getpass
import os

REPO_URL = "github.com/norewyx0205/vlm-event-boundary.git"
REPO_DIR = "/content/vlm-event-boundary"

if not os.path.exists(REPO_DIR):
    token = os.environ["GH_TOKEN"]
    if token:
        !git clone https://{token}@{REPO_URL} {REPO_DIR}
    else:
        !git clone https://{REPO_URL} {REPO_DIR}
else:
    print("Repository already exists; pulling latest changes...")
    %cd {REPO_DIR}
    !git pull


Repository already exists; pulling latest changes...
/content/vlm-event-boundary
Already up to date.


In [8]:
%cd /content/vlm-event-boundary
!ls

/content/vlm-event-boundary
analysis			notebooks    scripts
baseline_boundary_videos	README.md    synthetic_boundary_videos
data				results
generate_2d_boundary_videos.py	run_eval.py


## Install Dependencies

These packages are needed for Qwen video input, video generation, and result analysis.


In [9]:
!pip install "transformers==5.9.0" accelerate "qwen-vl-utils==0.0.14" "decord==0.6.0" opencv-python imageio-ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 103.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 128.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 73.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.10.2
    Uninstalling transformers-5.10.2:
      Successfully uninstalled transformers-5.10.2


In [10]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    torch.set_default_device("cuda")


CUDA available: True


## Configuration

Qwen3-VL is the default model for the ladder experiment. You can change the model string here if needed.


In [11]:
MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"
MODEL_REVISION = ""  # Set this to the model commit hash recorded in config.json to freeze future runs.
EVAL_SEED = 42
ATTN_IMPLEMENTATION = "eager"
RESULT_DIR = "/content/vlm-event-boundary/results"

BASELINE_ANNOTATION = "/content/vlm-event-boundary/baseline_boundary_videos/annotations.jsonl"
SYNTHETIC_ANNOTATION = "/content/vlm-event-boundary/synthetic_boundary_videos/annotations.jsonl"
LADDER_ROOT = "/content/vlm-event-boundary/data/ladder_v2"
DATASET_VERSION = "ladder_v2"

## Generate Baseline and Synthetic Reference Datasets

This regenerates the legacy simple baseline and the harder synthetic reference set. These are kept as reference points outside the 6-level ladder.

In [12]:
# Video generation is disabled for normal evaluation runs.
# Uncomment the next line when the baseline/synthetic datasets need to be regenerated.
# !python generate_2d_boundary_videos.py --dataset all

## Baseline Sanity Check

This keeps the earlier simple baseline. It should verify that Qwen3 can solve the easy before/after task.


In [13]:
from pathlib import Path

for name, annotation in [
    ("baseline", BASELINE_ANNOTATION),
    ("synthetic", SYNTHETIC_ANNOTATION),
]:
    path = Path(annotation)
    print(f"{name} annotation exists:", path.exists())
    if path.exists():
        print(f"{name} eval rows:", sum(1 for _ in open(path)))
        print(f"{name} videos:", len(list((path.parent / "videos").glob("*.mp4"))))
    else:
        print(f"{name} files not found; run the generation cell above.")

baseline annotation exists: True
baseline eval rows: 40
baseline videos: 20
synthetic annotation exists: True
synthetic eval rows: 240
synthetic videos: 120


In [14]:
!python scripts/run_eval.py \
  --annotation_path "$BASELINE_ANNOTATION" \
  --model_name "$MODEL_NAME" \
  --model_revision "$MODEL_REVISION" \
  --seed "$EVAL_SEED" \
  --deterministic \
  --attn_implementation "$ATTN_IMPLEMENTATION" \
  --dataset_name baseline_qwen3_sanity_check \
  --output_dir "$RESULT_DIR"

config.json: 100% 1.47k/1.47k [00:00<00:00, 2.59MB/s]
model.safetensors.index.json: 100% 67.8k/67.8k [00:00<00:00, 56.6MB/s]
Fetching 4 files: 100% 4/4 [00:45<00:00, 11.36s/it]
Download complete: 100% 17.5G/17.5G [00:45<00:00, 510MB/s]                
Loading weights:   0% 0/750 [00:00<?, ?it/s]
Loading weights:   0% 1/750 [00:01<12:30,  1.00s/it]
Loading weights:   3% 26/750 [00:01<00:22, 32.03it/s]
Loading weights:   5% 41/750 [00:01<00:15, 45.49it/s]
Loading weights:   7% 54/750 [00:01<00:12, 56.13it/s]
Loading weights:   9% 66/750 [00:01<00:10, 64.74it/s]
Loading weights:  10% 77/750 [00:01<00:09, 69.83it/s]
Loading weights:  12% 87/750 [00:01<00:09, 72.56it/s]
Loading weights:  13% 97/750 [00:01<00:08, 75.10it/s]
Loading weights:  14% 106/750 [00:02<00:08, 75.39it/s]
Loading weights:  15% 115/750 [00:02<00:08, 77.98it/s]
Loading weights:  17% 125/750 [00:02<00:07, 80.70it/s]
Loading weights:  18% 134/750 [00:02<00:07, 82.08it/s]
Loading weights:  20% 147/750 [00:02<00:07, 80.87it/

## Synthetic Hard Reference Evaluation

This evaluates the legacy harder synthetic set so it can be compared with the simple baseline and the ladder levels.

In [15]:
!python scripts/run_eval.py \
  --annotation_path "$SYNTHETIC_ANNOTATION" \
  --model_name "$MODEL_NAME" \
  --model_revision "$MODEL_REVISION" \
  --seed "$EVAL_SEED" \
  --deterministic \
  --attn_implementation "$ATTN_IMPLEMENTATION" \
  --dataset_name synthetic_qwen3_reference \
  --output_dir "$RESULT_DIR"

Loading weights: 100% 750/750 [00:05<00:00, 131.65it/s]

Running synthetic_qwen3_reference from /content/vlm-event-boundary/synthetic_boundary_videos/annotations.jsonl
Processing sample_001_low_boundary.mp4::prompt_original
qwen-vl-utils using torchcodec to read video.
sample_001_low_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing sample_001_low_boundary.mp4::prompt_swapped
sample_001_low_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing sample_002_low_boundary.mp4::prompt_original
sample_002_low_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing sample_002_low_boundary.mp4::prompt_swapped
sample_002_low_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing sample_003_low_boundary.mp4::prompt_original
sample_003_low_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing sample_003_low_boundary.mp4::prompt_swapped
sample_003_low_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing samp

## Generate 6-Level Ladder Dataset

This creates the 6-level `data/ladder_v2/level_*` dataset with evaluation-level mirrored annotations. Re-run this cell when generation parameters change.


In [16]:
# Video generation is disabled for normal evaluation runs.
# Uncomment this command block when the ladder dataset needs to be regenerated.
# !python scripts/generate_ladder_dataset.py \
#   --dataset_version "$DATASET_VERSION" \
#   --samples_per_level 30 \
#   --output_root "$LADDER_ROOT" \
#   --seed 42


## Check Ladder Dataset

Each level should contain 30 base samples × 4 boundary conditions × 2 mirrored prompts = 240 evaluation rows. The full ladder has 6 levels.


In [17]:
from pathlib import Path

for ann in sorted(Path(LADDER_ROOT).glob("level_*/annotations.jsonl")):
    video_count = len(list((ann.parent / "videos").glob("*.mp4")))
    row_count = sum(1 for _ in open(ann))
    print(ann.parent.name, "videos=", video_count, "eval_rows=", row_count)


level_1_simple videos= 120 eval_rows= 240
level_2_randomized videos= 120 eval_rows= 240
level_3_non_target_static_distractors videos= 120 eval_rows= 240
level_4_target_like_static_distractors videos= 120 eval_rows= 240
level_5_target_like_moving_distractors videos= 120 eval_rows= 240
level_6_hard_temporal_interference videos= 120 eval_rows= 240


## Qwen3 Ladder Smoke Test

Run a tiny subset before launching the full ladder evaluation.


In [18]:
!python scripts/run_eval.py \
  --annotation_path "$LADDER_ROOT/level_1_simple/annotations.jsonl" \
  --model_name "$MODEL_NAME" \
  --model_revision "$MODEL_REVISION" \
  --seed "$EVAL_SEED" \
  --deterministic \
  --attn_implementation "$ATTN_IMPLEMENTATION" \
  --dataset_name smoke_ladder_v2_level_1_simple_qwen3 \
  --output_dir "$RESULT_DIR" \
  --max_samples 4

Loading weights: 100% 750/750 [00:05<00:00, 133.96it/s]

Running smoke_ladder_v2_level_1_simple_qwen3 from /content/vlm-event-boundary/data/ladder_v2/level_1_simple/annotations.jsonl
Processing level_1_sample_001_low_boundary_original
qwen-vl-utils using torchcodec to read video.
level_1_sample_001_low_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing level_1_sample_001_low_boundary_swapped
level_1_sample_001_low_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing level_1_sample_001_temporal_boundary_original
level_1_sample_001_temporal_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing level_1_sample_001_temporal_boundary_swapped
level_1_sample_001_temporal_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'

Saved raw results to /content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/smoke_ladder_v2_level_1_simple_qwen3/20260612_204701/raw_results.jsonl
Saved summary to /content/vlm-event-boundary/results/Qwen_Qwen3-VL

## Run Qwen3 on All 6 Ladder Levels

This is the main ladder experiment. Results are saved under `results/<safe_model_name>/<dataset_name>/<timestamp>/`.


In [19]:
!python scripts/run_eval.py \
  --annotation_root "$LADDER_ROOT" \
  --model_name "$MODEL_NAME" \
  --model_revision "$MODEL_REVISION" \
  --seed "$EVAL_SEED" \
  --deterministic \
  --attn_implementation "$ATTN_IMPLEMENTATION" \
  --dataset_name_prefix "$DATASET_VERSION"_ \
  --output_dir "$RESULT_DIR"


Loading weights: 100% 750/750 [00:05<00:00, 133.62it/s]

Running ladder_v2_level_1_simple from /content/vlm-event-boundary/data/ladder_v2/level_1_simple/annotations.jsonl
Processing level_1_sample_001_low_boundary_original
qwen-vl-utils using torchcodec to read video.
level_1_sample_001_low_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing level_1_sample_001_low_boundary_swapped
level_1_sample_001_low_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing level_1_sample_001_temporal_boundary_original
level_1_sample_001_temporal_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing level_1_sample_001_temporal_boundary_swapped
level_1_sample_001_temporal_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing level_1_sample_001_visual_boundary_original
level_1_sample_001_visual_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing level_1_sample_001_visual_boundary_swapped
level_1_sample_001_visual_boundary.mp4 p

## Analyze Ladder Results

This aggregates all Qwen3 ladder runs and treats prompt Accuracy and Strict both-correct pair accuracy as co-primary metrics. It produces both 6-level curves, direct Accuracy-vs-Strict comparisons, paired boundary comparisons, and swap-consistency diagnostics.



In [20]:
ANALYSIS_DIR = f"analysis/{DATASET_VERSION}_ladder"

!python scripts/analyze_results.py \
  --input "$RESULT_DIR" \
  --dataset_name_prefix "ladder_v2_level_" \
  --output_dir "$ANALYSIS_DIR" \
  --plots

Analyzed 1440 rows from 6 raw result file(s).
Saved analysis to analysis/ladder_v2_ladder


## Inspect Saved Files


In [21]:
!find "$RESULT_DIR" -maxdepth 4 -type f | sort | tail -60
!find analysis -maxdepth 2 -type f | sort


/content/vlm-event-boundary/results/qwen2vl_2b/ladder_v1/.gitkeep
/content/vlm-event-boundary/results/qwen3vl/ladder_v1/.gitkeep
/content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/baseline_qwen3_sanity_check/20260612_203415/config.json
/content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/baseline_qwen3_sanity_check/20260612_203415/raw_results.jsonl
/content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/baseline_qwen3_sanity_check/20260612_203415/summary.json
/content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/ladder_v2_level_1_simple/20260612_204724/config.json
/content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/ladder_v2_level_1_simple/20260612_204724/raw_results.jsonl
/content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/ladder_v2_level_1_simple/20260612_204724/summary.json
/content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/ladder_v2_level_2_randomized/20260612_204724/config.json
/content/vlm-event-boundary/results/Qwe

## Level 5 Feature-Ablation Pilot

This pilot compares four structurally paired Level 5 variants: full, shape-only, color-only, and size-only. It also includes a separate size-only 2x2 stress pilot crossing absolute target size with distractor count.


In [22]:
ABLATION_VERSION = "l5_feature_ablation_v1"
ABLATION_ROOT = f"/content/vlm-event-boundary/data/{ABLATION_VERSION}"
ABLATION_ANALYSIS_DIR = f"/content/vlm-event-boundary/analysis/{ABLATION_VERSION}"
SIZE_STRESS_ROOT = f"{ABLATION_ROOT}/size_stress_pilot"
SIZE_STRESS_ANALYSIS_DIR = f"{ABLATION_ANALYSIS_DIR}_size_stress"
ABLATION_VARIANTS = ["L5_full", "L5_shape_only", "L5_color_only", "L5_size_only"]


### Generate And Validate Paired Stimuli


In [23]:
# Video generation is disabled for normal evaluation runs.
# Uncomment this command block when the L5 ablation dataset needs to be regenerated.
# !python scripts/generate_l5_feature_ablation.py \
#   --dataset_version "$ABLATION_VERSION" \
#   --samples_per_variant 30 \
#   --size_stress_samples_per_cell 10 \
#   --output_root "$ABLATION_ROOT" \
#   --seed 42
#
# !python scripts/check_l5_feature_ablation.py --root "$ABLATION_ROOT"


### Run Qwen3 Evaluation

Each variant retains all four boundary conditions and original/swapped mirrored prompts.


In [24]:
!python scripts/run_eval.py \
  --annotation_root "$ABLATION_ROOT" \
  --model_name "$MODEL_NAME" \
  --model_revision "$MODEL_REVISION" \
  --seed "$EVAL_SEED" \
  --deterministic \
  --attn_implementation "$ATTN_IMPLEMENTATION" \
  --dataset_name_prefix "$ABLATION_VERSION"_main_ \
  --output_dir "$RESULT_DIR"


Loading weights: 100% 750/750 [00:05<00:00, 129.62it/s]

Running l5_feature_ablation_v1_main_L5_color_only from /content/vlm-event-boundary/data/l5_feature_ablation_v1/L5_color_only/annotations.jsonl
Processing l5_color_only_sample_001_low_boundary_original
qwen-vl-utils using torchcodec to read video.
l5_color_only_sample_001_low_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing l5_color_only_sample_001_low_boundary_swapped
l5_color_only_sample_001_low_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing l5_color_only_sample_001_temporal_boundary_original
l5_color_only_sample_001_temporal_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing l5_color_only_sample_001_temporal_boundary_swapped
l5_color_only_sample_001_temporal_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing l5_color_only_sample_001_visual_boundary_original
l5_color_only_sample_001_visual_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Proce

### Run Size-Only 2x2 Stress Pilot

This evaluates 10 base samples in each of large/few, large/many, small/few, and small/many, for 320 prompt evaluations in total.


In [25]:
!python scripts/run_eval.py \
  --annotation_root "$SIZE_STRESS_ROOT" \
  --model_name "$MODEL_NAME" \
  --model_revision "$MODEL_REVISION" \
  --seed "$EVAL_SEED" \
  --deterministic \
  --attn_implementation "$ATTN_IMPLEMENTATION" \
  --dataset_name_prefix "$ABLATION_VERSION"_size_stress_ \
  --output_dir "$RESULT_DIR"


Loading weights: 100% 750/750 [00:05<00:00, 131.96it/s]

Running l5_feature_ablation_v1_size_stress_L5_size_only_large_few from /content/vlm-event-boundary/data/l5_feature_ablation_v1/size_stress_pilot/L5_size_only_large_few/annotations.jsonl
Processing l5_size_only_large_few_sample_001_low_boundary_original
qwen-vl-utils using torchcodec to read video.
l5_size_only_large_few_sample_001_low_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing l5_size_only_large_few_sample_001_low_boundary_swapped
l5_size_only_large_few_sample_001_low_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing l5_size_only_large_few_sample_001_temporal_boundary_original
l5_size_only_large_few_sample_001_temporal_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing l5_size_only_large_few_sample_001_temporal_boundary_swapped
l5_size_only_large_few_sample_001_temporal_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing l5_size_only_large_few_sample_

### Analyze Feature, Boundary, And Position Effects

The analysis uses the latest run for each variant and writes prompt accuracy, strict mirrored-pair accuracy, the accuracy-strict gap `d`, position-sensitive pair rates, paired comparisons, swap consistency, and report-ready plots for the main research questions.


In [26]:
!python scripts/analyze_results.py \
  --input "$RESULT_DIR" \
  --dataset_name_prefix "$ABLATION_VERSION"_main_ \
  --latest_per_dataset \
  --output_dir "$ABLATION_ANALYSIS_DIR" \
  --plots

!find "$ABLATION_ANALYSIS_DIR" -maxdepth 1 -type f | sort


Analyzed 960 rows from 4 raw result file(s).
Saved analysis to /content/vlm-event-boundary/analysis/l5_feature_ablation_v1
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1/accuracy_by_correct_option.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1/accuracy_by_difficulty_condition.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1/accuracy_by_difficulty_condition.png
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1/accuracy_by_difficulty.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1/accuracy_by_feature_variant_condition.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1/accuracy_by_feature_variant_condition.png
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1/accuracy_by_feature_variant_correct_option.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1/accuracy_by_feature_variant.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1/accuracy_by_feature_variant_prompt_variant

### Analyze Size And Crowding Effects

This produces cell-level accuracy, strict mirrored-pair accuracy, boundary-condition plots, and the large-vs-small, few-vs-many, and interaction estimates.


In [27]:
!python scripts/analyze_results.py \
  --input "$RESULT_DIR" \
  --dataset_name_prefix "$ABLATION_VERSION"_size_stress_ \
  --latest_per_dataset \
  --output_dir "$SIZE_STRESS_ANALYSIS_DIR" \
  --plots

!find "$SIZE_STRESS_ANALYSIS_DIR" -maxdepth 1 -type f | sort


Analyzed 320 rows from 4 raw result file(s).
Saved analysis to /content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_stress
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_stress/accuracy_by_correct_option.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_stress/accuracy_by_difficulty_condition.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_stress/accuracy_by_difficulty_condition.png
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_stress/accuracy_by_difficulty.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_stress/accuracy_by_prompt_variant.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_stress/accuracy_by_size_scene_condition.csv
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_stress/accuracy_by_size_scene_condition.png
/content/vlm-event-boundary/analysis/l5_feature_ablation_v1_size_stress/accuracy_by_size_scene_correct_option.csv
/content/v

## Download Timestamped Experiment Archive

Run this after an experiment. It packages all saved evaluation results, analyses, and ablation annotations into a timestamped ZIP and downloads it to your local computer. Videos are excluded to keep the archive manageable.


In [28]:
from datetime import datetime
import json
from pathlib import Path
import zipfile
from google.colab import files

project_root = Path("/content/vlm-event-boundary")
archive_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archive_path = Path("/content") / f"vlm_event_boundary_results_{archive_timestamp}.zip"
manifest = {
    "created_at": archive_timestamp,
    "model_name": MODEL_NAME,
    "model_revision": MODEL_REVISION or None,
    "eval_seed": EVAL_SEED,
    "deterministic": True,
    "attention_implementation": ATTN_IMPLEMENTATION,
    "ladder_version": DATASET_VERSION,
    "ablation_version": globals().get("ABLATION_VERSION"),
}

with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr("archive_manifest.json", json.dumps(manifest, indent=2))
    for folder in [Path(RESULT_DIR), project_root / "analysis"]:
        if folder.exists():
            for path in folder.rglob("*"):
                if path.is_file():
                    archive.write(path, path.relative_to(project_root))
    ablation_root = Path(globals().get("ABLATION_ROOT", ""))
    if ablation_root.exists():
        for pattern in ["annotations.jsonl", "config.json", "README.md"]:
            for path in ablation_root.rglob(pattern):
                archive.write(path, path.relative_to(project_root))

print(f"Created {archive_path} ({archive_path.stat().st_size / 1024 / 1024:.1f} MB)")
files.download(str(archive_path))


Created /content/vlm_event_boundary_results_20260612_221809.zip (0.5 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>